### Chapter 6.7 - GPUs

GPUs accelerate tensor computation when available. This notebook explains devices with code that remains inspectable and safe on machines without a GPU.


In [ ]:
from pathlib import Path
import tempfile
import torch


### 6.7.1 Computing Devices

#### 1. Intuition

A computing device is hardware where tensors and operations live. Common devices are CPU and GPU.

A CPU is general-purpose. A GPU is designed for many parallel numerical operations.

#### 2. Why this exists

Deep learning can be much faster on GPUs because tensor operations often contain many independent arithmetic tasks.


#### 3. Examples

Check whether CUDA GPU support is available.


In [ ]:
has_cuda = torch.cuda.is_available()
device = torch.device("cuda" if has_cuda else "cpu")
has_cuda, device


Count visible CUDA devices safely.


In [ ]:
count = torch.cuda.device_count()
count


#### 4. Step-by-step breakdown

`torch.cuda.is_available()` returns whether PyTorch can use CUDA.

`torch.device(...)` creates a device object.

The conditional chooses GPU only when available.

`device_count()` reports how many CUDA devices PyTorch sees.

#### 5. Connection to ML systems

Training scripts often choose a device at startup and move tensors and models there.

#### 6. Common confusion points

- CUDA is NVIDIA's GPU computing platform.
- A machine can have PyTorch installed without GPU support.
- Device checks should be conditional, not assumed.
- CPU examples should still run conceptually without GPU hardware.


### 6.7.2 Tensors and GPUs

#### 1. Intuition

A tensor lives on one device at a time. Operations usually require tensors to be on the same device.

Moving a tensor means copying its data to another device.

#### 2. Why this exists

Device mismatch is a common PyTorch error. A model on GPU cannot directly compute with input tensors left on CPU.


#### 3. Examples

Create a tensor on the selected device.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.tensor([1.0, 2.0], device=device)
x.device


Move a tensor back to CPU.


In [ ]:
x_cpu = x.to("cpu")
x_cpu.device


#### 4. Step-by-step breakdown

The device is chosen safely based on availability.

`torch.tensor(..., device=device)` creates the tensor directly on that device.

`.to("cpu")` moves or keeps the tensor on CPU.

The `.device` attribute reports where the tensor lives.

#### 5. Connection to ML systems

Data batches must be moved to the same device as the model before the forward pass.

#### 6. Common confusion points

- Device movement can copy data and cost time.
- Tensor device and tensor dtype are different concepts.
- Operations across different devices usually fail.
- Keep device handling explicit in training code.


### 6.7.3 Neural Networks and GPUs

#### 1. Intuition

A neural network module also lives on a device through its parameters.

Moving a module to a device moves its parameters and buffers.

#### 2. Why this exists

Model parameters and input tensors must be on the same device for computation.


#### 3. Examples

Move a small network and input to the selected device.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = torch.nn.Linear(2, 1).to(device)
X = torch.randn(3, 2).to(device)
y_hat = net(X)
y_hat.device


#### 4. Step-by-step breakdown

The selected device is GPU if available, otherwise CPU.

`.to(device)` moves the module parameters.

The input tensor is moved to the same device.

The output is produced on that device.

#### 5. Connection to ML systems

Training loops usually move each minibatch to the model's device before computing predictions.

#### 6. Common confusion points

- Moving the model is not enough; move input data too.
- Outputs are usually on the same device as the computation.
- Convert to CPU before NumPy conversion.
- Device-safe code should not assume GPU availability.


### 6.7.4 Summary

#### 1. Intuition

Device management controls where tensors and models are stored and computed.

Correct device placement is required for GPU acceleration and for avoiding mismatch errors.

#### 2. Why this exists

As models grow, hardware placement becomes part of the training system design.


#### 3. Examples

A device-safe training setup checklist.


In [ ]:
checklist = [
    "choose device",
    "move model",
    "move each batch",
    "move outputs to CPU for logging if needed",
]
checklist


#### 4. Step-by-step breakdown

The checklist names common device-handling steps.

Choosing a device happens once near startup.

Batches are moved during training.

Logging sometimes needs CPU values.

#### 5. Connection to ML systems

Good device handling lets the same code work on CPU laptops and GPU servers.

#### 6. Common confusion points

- Hardware availability varies by machine.
- Device checks should be explicit.
- CPU fallback is useful for teaching and debugging.
- GPU speedups require enough work to offset transfer costs.


### 6.7.5 Exercises

#### 1. Intuition

These exercises practice device-safe code without requiring a GPU.

#### 2. Why this exists

The habit is to write code that adapts to available hardware.


#### 3. Examples

Exercise 1: choose a safe device.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


Exercise 2: create a tensor on that device.


In [ ]:
x = torch.ones(2, device=device)
x.device


#### 4. Step-by-step breakdown

Exercise 1 chooses GPU only when available.

Exercise 2 creates a tensor directly on the chosen device.

Both examples run conceptually on CPU-only machines.

#### 5. Connection to ML systems

Device-safe code is the starting point for portable training scripts.

#### 6. Common confusion points

- Never assume every machine has a GPU.
- Always check device placement when errors mention device mismatch.
- Move data and model together.
- Keep examples hardware-safe when teaching.
